In [3]:
from pygris.data import get_census
import pygris
import numpy as np

In [1]:
# DP03_0062E: Estimate!!INCOME AND BENEFITS (IN 2019 INFLATION-ADJUSTED DOLLARS)!!Total households!!Median household income (dollars)
# DP04_0089E: Estimate!!VALUE!!Owner-occupied units!!Median (dollars)
# DP04_0134E: Estimate!!GROSS RENT!!Occupied units paying rent!!Median (dollars)
# DP05_0001E: Estimate!!SEX AND AGE!!Total population
variables = ['DP03_0062E', 'DP04_0089E', 'DP04_0134E', 'DP05_0001E']

In [4]:
ca_gentrification_vars_23 = get_census(dataset = "acs/acs5/profile",
                        variables = variables,
                        year = 2023,
                        params = {
                          "for": "tract:*",
                          "in": "state:06"},
                        guess_dtypes = True,
                        return_geoid = True)

In [18]:
ca_gentrification_vars_23 = ca_gentrification_vars_23.rename({'DP03_0062E': 'medhinc23', 
                                                              'DP04_0089E': 'medhval23', 
                                                              'DP04_0134E': 'medrent23', 
                                                              'DP05_0001E': 'totpop23'}, axis=1)

In [11]:
ca_gentrification_vars_19 = get_census(dataset = "acs/acs5/profile",
                        variables = variables,
                        year = 2019,
                        params = {
                          "for": "tract:*",
                          "in": "state:06"},
                        guess_dtypes = True,
                        return_geoid = True)

In [19]:
ca_gentrification_vars_19 = ca_gentrification_vars_19.rename({'DP03_0062E': 'medhinc19', 
                                                              'DP04_0089E': 'medhval19', 
                                                              'DP04_0134E': 'medrent19', 
                                                              'DP05_0001E': 'totpop19'}, axis=1)

In [48]:
sd_tracts = pygris.tracts(state = "CA", county = "San Diego", cb = True,
                    year = 2023, cache = True)

Using FIPS code '06' for input 'CA'
Using FIPS code '073' for input 'San Diego'


In [49]:
sd_tracts19 = pygris.tracts(state = "CA", county = "San Diego", cb = True,
                    year = 2019, cache = True)

Using FIPS code '06' for input 'CA'
Using FIPS code '073' for input 'San Diego'


In [50]:
tractid19 = []
for pt in sd_tracts.representative_point():
    tractid19.append(sd_tracts19[sd_tracts19.intersects(pt)]['GEOID'].iloc[0])

In [51]:
sd_tracts['GEOID19'] = tractid19

In [52]:
sd_tracts = sd_tracts.merge(ca_gentrification_vars_23, how = "inner", on = "GEOID")

In [55]:
sd_tracts = sd_tracts.merge(ca_gentrification_vars_19, how = "inner", left_on = "GEOID19", right_on='GEOID')

In [69]:
sd_tracts = sd_tracts.to_crs(2230)
sd_tracts19 = sd_tracts19.to_crs(2230)

In [74]:
THRESHOLD = 0.1

In [75]:
def is_close(num1, num2):
  """Checks if num2 is within 10% of num1."""
  return num1 * (1 - THRESHOLD) < num2 < num1 * (1 + THRESHOLD)

In [97]:
sd_tracts['totpop19_smooth'] = 0

In [102]:
sd_dup19 = sd_tracts[sd_tracts.duplicated('GEOID19', keep=False)]

for geoid in sd_dup19['GEOID19']:
    area19 = sd_tracts19[sd_tracts19['GEOID'] == geoid].geometry.area.iloc[0]
    area23 = sd_tracts[sd_tracts['GEOID19'] == geoid].geometry.area
    if not is_close(sum(area23), area19):
        print(geoid)
        print("!!!")
        print(sum(area23)/area19)

    area_r = sum(area23)/area19
    area_r*area23/area19
    smoothed_pop = np.floor(area23/area_r/area19*sd_tracts[sd_tracts['GEOID19'] == geoid]['totpop19']).astype(int)
    sd_tracts.loc[smoothed_pop.index, 'totpop19_smooth'] = smoothed_pop

06073005100
!!!
1.917489098052324
06073013311
!!!
1.1338903153493542
06073005100
!!!
1.917489098052324
06073013311
!!!
1.1338903153493542
06073005100
!!!
1.917489098052324


In [103]:
sd_tracts[sd_tracts['GEOID19'] == '06073005100']

,STATEFP,COUNTYFP,TRACTCE,GEOIDFQ,GEOID_x,NAME,NAMELSAD,STUSPS,NAMELSADCO,STATE_NAME,...,medhinc23,medhval23,medrent23,totpop23,medhinc19,medhval19,medrent19,totpop19,GEOID_y,totpop19_smooth
37,06,073,005102,1400000US06073005102,06073005102,51.02,Census Tract 51.02,CA,San Diego County,California,...,85541.0,661300.0,2423.0,4673,46658.0,416800.0,1064.0,7702,06073005100,548
312,06,073,005103,1400000US06073005103,06073005103,51.03,Census Tract 51.03,CA,San Diego County,California,...,24125.0,NaN,876.0,3045,46658.0,416800.0,1064.0,7702,06073005100,6585
655,06,073,005101,1400000US06073005101,06073005101,51.01,Census Tract 51.01,CA,San Diego County,California,...,64155.0,NaN,2422.0,3618,46658.0,416800.0,1064.0,7702,06073005100,567


In [78]:
area_r = sum(area23)/area19

In [81]:
area_r

np.float64(1.0056863469524115)

In [95]:
smoothed_pop = np.floor(area23/area_r/area19*sd_tracts[sd_tracts['GEOID19'] == geoid]['totpop19']).astype(int)

34     0
604    0
667    0
708    0
Name: totpop19_smooth, dtype: int64

34     3246
604    3303
667    2211
708    7602
dtype: int64

In [71]:
sum(sd_tracts[sd_tracts['GEOID19'] == geoid].geometry.area)

168286628.98998648

np.float64(167335103.5339746)

In [64]:
sd_tracts.geometry

0      POLYGON ((-117.06161 32.66388, -117.06041 32.6...
1      POLYGON ((-117.08407 32.61716, -117.07996 32.6...
2      POLYGON ((-116.79025 32.90675, -116.78983 32.9...
3      POLYGON ((-116.97883 32.78368, -116.97488 32.7...
4      POLYGON ((-117.14252 32.71944, -117.14031 32.7...
                             ...                        
731    POLYGON ((-117.29657 33.18563, -117.29636 33.1...
732    POLYGON ((-117.07856 32.63784, -117.07648 32.6...
733    POLYGON ((-117.16573 32.72719, -117.163 32.727...
734    POLYGON ((-117.07083 32.67638, -117.06657 32.6...
735    POLYGON ((-117.25969 33.37671, -117.25361 33.3...
Name: geometry, Length: 736, dtype: geometry